# 03 — Baseline: Area Filter + Douglas-Peucker

Classical generalization pipeline as benchmark:
1. Remove BGT polygons below an area threshold
2. Simplify remaining geometries with Douglas-Peucker

Metrics saved here are reused in notebook 05 to compare against the GNN.

Run **01_data_prep.ipynb** first.


## 0. CONFIG

In [ ]:
from pathlib import Path

CONFIG = {
    "output_root": Path("processed"),
    "crs":         "EPSG:28992",

    # ── Baseline parameters ───────────────────────────────
    # Polygons smaller than this are removed (m²)
    "min_area_m2":    25.0,

    # Douglas-Peucker tolerance in metres
    "dp_tolerance_m": 1.0,

    # Evaluate on this many test tiles (None = all)
    "eval_n_tiles":   None,
}

## 1. Imports

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)
print("Imports OK")

## 2. Baseline and metric functions

In [ ]:
def generalize_baseline(bgt: gpd.GeoDataFrame, config: dict) -> gpd.GeoDataFrame:
    """Area filter → Douglas-Peucker simplification."""
    out = bgt[bgt.geometry.area >= config["min_area_m2"]].copy()
    out["geometry"] = out.geometry.simplify(config["dp_tolerance_m"], preserve_topology=True)
    out = out[out.geometry.is_valid & ~out.geometry.is_empty & (out.geometry.area > 0)]
    return out


def iou_union(pred: gpd.GeoDataFrame, target: gpd.GeoDataFrame) -> float:
    """IoU between the dissolved union of pred and target polygons."""
    try:
        pu = pred.geometry.union_all()
        tu = target.geometry.union_all()
        inter = pu.intersection(tu).area
        union = pu.union(tu).area
        return inter / union if union > 0 else 0.0
    except Exception:
        return 0.0


def evaluate_tile(pred: gpd.GeoDataFrame, target: gpd.GeoDataFrame) -> dict:
    """Compute IoU, Hausdorff, count ratio, and vertex reduction for one tile."""
    if len(pred) == 0 or len(target) == 0:
        return {"iou": 0.0, "hausdorff": float("nan"),
                "count_ratio": 0.0, "vertex_reduction": float("nan")}

    pu = pred.geometry.union_all()
    tu = target.geometry.union_all()

    iou_val = iou_union(pred, target)

    try:
        hausdorff = pu.hausdorff_distance(tu)
    except Exception:
        hausdorff = float("nan")

    count_ratio = len(pred) / max(len(target), 1)

    def verts(gdf):
        return sum(
            len(g.exterior.coords) if g.geom_type == "Polygon"
            else sum(len(p.exterior.coords) for p in g.geoms)
            for g in gdf.geometry
        )
    vertex_reduction = 1.0 - verts(pred) / max(verts(target), 1)

    return {
        "iou":              round(iou_val, 4),
        "hausdorff":        round(hausdorff, 2),
        "count_ratio":      round(count_ratio, 3),
        "vertex_reduction": round(vertex_reduction, 4),
    }


print("Functions defined.")

## 3. Run baseline on test tiles

In [ ]:
with open(CONFIG["output_root"] / "tile_index.json") as f:
    tile_index = json.load(f)

test_tiles = tile_index["test"]
if CONFIG["eval_n_tiles"]:
    test_tiles = test_tiles[:CONFIG["eval_n_tiles"]]

print(f"Evaluating baseline on {len(test_tiles)} test tiles...")
print(f"  min_area    = {CONFIG['min_area_m2']} m²")
print(f"  dp_tolerance= {CONFIG['dp_tolerance_m']} m")

results = []
for i, tile in enumerate(test_tiles):
    bgt  = gpd.read_file(tile["bgt"])
    brt  = gpd.read_file(tile["brt"])
    pred = generalize_baseline(bgt, CONFIG)
    m    = evaluate_tile(pred, brt)
    m["tile"] = tile["bgt"]
    m["city"] = tile["city"]
    results.append(m)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{len(test_tiles)}")

results_df = pd.DataFrame(results)
print(f"\nDone — {len(results_df)} tiles evaluated.")

print("\nBaseline summary:")
print(results_df[["iou", "hausdorff", "count_ratio", "vertex_reduction"]].describe().round(4).to_string())

results_df.to_csv(CONFIG["output_root"] / "baseline_results.csv", index=False)
print("\nResults saved.")

## 4. Visualize baseline on one tile

In [ ]:
# Pick tile closest to median IoU
med_idx  = (results_df["iou"] - results_df["iou"].median()).abs().idxmin()
example  = test_tiles[med_idx]
bgt_ex   = gpd.read_file(example["bgt"])
brt_ex   = gpd.read_file(example["brt"])
pred_ex  = generalize_baseline(bgt_ex, CONFIG)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
bgt_ex.plot( ax=axes[0], color="steelblue",  edgecolor="white", linewidth=0.3)
pred_ex.plot(ax=axes[1], color="darkorange", edgecolor="white", linewidth=0.3)
brt_ex.plot( ax=axes[2], color="tomato",     edgecolor="white", linewidth=0.3)

axes[0].set_title(f"BGT input ({len(bgt_ex)} polygons)")
axes[1].set_title(f"Baseline output ({len(pred_ex)} polygons)")
axes[2].set_title(f"BRT target ({len(brt_ex)} polygons)")
for ax in axes: ax.set_axis_off()

m = results_df.loc[med_idx]
plt.suptitle(f"Baseline — IoU: {m['iou']:.3f}  Hausdorff: {m['hausdorff']:.1f}m  Count ratio: {m['count_ratio']:.2f}")
plt.tight_layout()
plt.savefig(CONFIG["output_root"] / "baseline_example.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nMean IoU          : {results_df['iou'].mean():.4f}")
print(f"Mean Hausdorff    : {results_df['hausdorff'].mean():.2f} m")
print(f"Mean count ratio  : {results_df['count_ratio'].mean():.3f}")
print("\nThese are the numbers your GNN needs to beat.")